In [13]:
import hashlib
import json
import subprocess
import time
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split


In [14]:
# ==========================================
# 1. Pipeline Configuration & Constants
# ==========================================
WORKSPACE = Path('./')
DATA_PATH = WORKSPACE / 'data' / 'manual_labels_with_hal_metadata.csv'
ARTIFACT_DIR = WORKSPACE / 'mpnet_sgd_artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)

MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'
CLASSIFIER_PATH = ARTIFACT_DIR / 'sgd_classifier.joblib'
ENCODER_PATH = ARTIFACT_DIR / 'mpnet_encoder'
SUBAXIS_PATH = ARTIFACT_DIR / 'subaxis_references.joblib'
METADATA_PATH = ARTIFACT_DIR / 'metadata.json'
EMBEDDING_CACHE_PATH = ARTIFACT_DIR / 'training_embeddings.npz'
REGISTERED_MODEL_NAME = "sequoia-mpnet-sgd-classifier"
BEST_METRIC_PATH = ARTIFACT_DIR / "best_metric.json"


RANDOM_STATE = 42
CANONICAL_LABELS = [
    'AI, cybersecurity and defense',
    'AI, environment and ocean',
    'Core AI',
    'No class',
]


SEQUOIA_AXES = {
    "Core AI": "core AI foundations learning secure trustworthy frugal embedded hybrid explainable AI hardware signal image language robotics theory human AI interaction regulation ethics",
    "AI, Cybersecurity and Defense": "AI cybersecurity defense systems infrastructures information intelligence digital influence security adversarial attacks privacy anomaly detection networks cyber defense disinformation robotics",
    "AI, Environment and Ocean": "AI environment ocean earth observation modeling climate remote sensing digital twins data assimilation uncertainty marine pollution maritime traffic ocean dynamics",
}

SEQUOIA_SAXES = {

    "Core AI": {

        "Theoretical and Conceptual Foundations of AI": [
            "generalization",
            "statistical bounds",
            "continual learning",
            "optimal transport",
            "long-tail distributions",
            "rare events",
            "time series",
            "graphs",
            "geometric deep learning",
        ],

        "Symbolic / Statistical Hybridization": [
            "hybrid AI",
            "neuro-symbolic AI",
            "knowledge graphs",
            "counterfactual explanations",
            "epistemic planning",
            "reasoning",
        ],

        "Formal Evaluation of ML Technologies": [
            "formal methods",
            "model checking",
            "certification",
            "certified robustness",
            "neural network verification",
            "static analysis",
        ],

        "Frugal AI, Edge AI and Hardware Architectures": [
            "frugal AI",
            "quantization",
            "sparsity",
            "AI accelerators",
            "model compression",
            "energy efficiency",
            "edge AI",
            "compilation",
        ],

        "Human–AI Interaction and Explainability": [
            "XAI",
            "explainability",
            "trust",
            "confidence measures",
            "human-AI interaction",
            "attention mechanisms",
            "SHAP",
            "interactive learning",
        ],

        "Acceptability, Responsibility, Regulation and Ethics": [
            "AI Act",
            "AI regulation",
            "responsibility",
            "bias",
            "AI ethics",
            "law",
            "organizations",
            "AI use",
            "acceptability",
        ],

        "Signal, Image and Language": [
            "signal processing",
            "computer vision",
            "natural language processing",
            "NLP",
            "large language models",
            "LLMs",
            "diffusion models",
            "multimodal AI",
            "vision-language models",
            "image compression",
        ],
    },


    "AI, Cybersecurity and Defense": {

        "AI for Cybersecurity": [
            "intrusion detection",
            "anomaly detection",
            "vulnerabilities",
            "obfuscation",
            "cyber defense",
            "threat intelligence",
            "logs",
            "security monitoring",
        ],

        "AI Security": [
            "adversarial attacks",
            "data poisoning",
            "model inversion",
            "model privacy",
            "model confidentiality",
            "hardware attacks",
            "side-channel attacks",
            "federated learning",
        ],

        "Intelligence and Information Security": [
            "information extraction",
            "disinformation",
            "deepfakes",
            "information manipulation",
            "OSINT",
            "open-source intelligence",
            "fake news",
            "digital influence",
        ],

        "Intelligent and Secure Networks": [
            "computer networks",
            "5G",
            "software-defined networking",
            "SDN",
            "network functions virtualization",
            "NFV",
            "graph neural networks for networks",
            "network orchestration",
            "QoS",
            "quality of service",
        ],

        "Autonomous Robotics and Interaction": [
            "robotics",
            "shared control",
            "robot perception",
            "planning",
            "robot swarms",
            "human-robot interaction",
            "drones",
            "UAVs",
            "underwater robots",
            "autonomous robots",
        ],
    },


    "AI, Environment and Ocean": {

        "Physics-Informed AI": [
            "physics-informed AI",
            "physics-informed neural networks",
            "PINN",
            "data assimilation",
            "conformal prediction",
            "uncertainty quantification",
            "surrogate models",
        ],

        "Observation, Remote Sensing and Data Integration": [
            "remote sensing",
            "SAR",
            "synthetic aperture radar",
            "hyperspectral imaging",
            "sonar",
            "sensors",
            "sensor fusion",
            "data fusion",
            "AIS",
            "weak signals",
            "Earth observation",
        ],

        "Ocean Digital Twin": [
            "digital twin",
            "ocean digital twin",
            "ocean dynamics",
            "oceanography",
            "ocean modeling",
            "ocean-atmosphere modeling",
            "marine extremes",
            "underwater mapping",
            "subsea mapping",
        ],
    },
}



# Set tracking URI and experiment[cite: 1]
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('sequoia-mpnet-sgd-continual')


with mlflow.start_run(run_name="test_write"):
    mlflow.log_param("test", "ok")
    mlflow.log_metric("value", 1.0)

print("MLflow write OK")

MLflow write OK


In [15]:
# 2. DVC Data Synchronization
print("🔄 Checking & pulling latest dataset version via DVC...")
try:
    subprocess.run(["dvc", "pull", str(DATA_PATH)], check=True, cwd=str(WORKSPACE))
    print("✅ DVC dataset pull completed successfully.")
except Exception as e:
    print(f"⚠️ DVC pull failed or skipped: {e}")

🔄 Checking & pulling latest dataset version via DVC...
⚠️ DVC pull failed or skipped: [Errno 2] No such file or directory: 'dvc'


In [16]:
# ==========================================
# 3. Load & Prepare Dataset
# ==========================================
df = pd.read_csv(DATA_PATH, dtype={'halId_s': str})
text_columns = ['title_s', 'abstract', 'keywords']
df['text_for_classification'] = (
    df[text_columns]
    .fillna('')
    .astype(str)
    .agg(' '.join, axis=1)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

print(f"Total dataset records: {len(df):,}")

# Filter valid canonical labels
df = df[df['manual_label'].isin(CANONICAL_LABELS)].reset_index(drop=True)

# Train / Test split for holdout evaluation[cite: 1]
X_train_df, X_test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df['manual_label'],
    random_state=RANDOM_STATE
)

print(f"Training set records: {len(X_train_df):,}")
print(f"Test set records: {len(X_test_df):,}")


Total dataset records: 1,404
Training set records: 1,123
Test set records: 281


In [17]:
# ==========================================
# 4. Encoder
# ==========================================
if ENCODER_PATH.exists():
    encoder = SentenceTransformer(str(Path.cwd() / ENCODER_PATH),local_files_only=True)
else:
    encoder = SentenceTransformer(MODEL_NAME)
    encoder.save(ENCODER_PATH)



print("⚡ Processing text embeddings (Checking cache)...")
embedding_started = time.perf_counter()

# Fingerprint the current training and testing texts to ensure cache validity[cite: 1]
train_texts = X_train_df['text_for_classification'].tolist()
test_texts = X_test_df['text_for_classification'].tolist()

fingerprint_data = {
    "model_name": MODEL_NAME,
    "normalize_embeddings": True,
    "train_texts": train_texts,
    "test_texts": test_texts,
}
current_fingerprint = hashlib.sha256(
    json.dumps(fingerprint_data, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
).hexdigest()

cache_hit = False
if EMBEDDING_CACHE_PATH.exists():
    with np.load(EMBEDDING_CACHE_PATH, allow_pickle=False) as cache:
        if str(cache.get("fingerprint", "")) == current_fingerprint:
            train_embeddings = cache["train_embeddings"]
            test_embeddings = cache["test_embeddings"]
            cache_hit = True

if cache_hit:
    print(f"✅ Loaded cached embeddings from {EMBEDDING_CACHE_PATH}")
else:
    print("🔄 Cache miss or dataset updated. Computing new embeddings...")
    train_embeddings = encoder.encode(
        train_texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    test_embeddings = encoder.encode(
        test_texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    # Save newly computed embeddings to disk[cite: 1]
    np.savez_compressed(
        EMBEDDING_CACHE_PATH,
        train_embeddings=train_embeddings,
        test_embeddings=test_embeddings,
        fingerprint=np.array(current_fingerprint)
    )
    print(f"💾 Saved updated embeddings cache to {EMBEDDING_CACHE_PATH}")

embedding_seconds = time.perf_counter() - embedding_started


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

⚡ Processing text embeddings (Checking cache)...
✅ Loaded cached embeddings from mpnet_sgd_artifacts/training_embeddings.npz


In [18]:
# ==========================================
# 5. Training Phase: Init (fit) vs Continual (partial_fit)
# ==========================================
fit_started = time.perf_counter()

if CLASSIFIER_PATH.exists():
    # --- CONTINUAL LEARNING MODE ---
    print("\n🧠 Existing model found: running incremental update (`partial_fit`)...")
    classifier = joblib.load(CLASSIFIER_PATH)
    
    classifier.partial_fit(
        train_embeddings,
        X_train_df['manual_label'],
        classes=np.array(CANONICAL_LABELS)
    )
    training_mode = "incremental_partial_fit"
else:
    # --- INITIAL TRAINING MODE ---
    print("\n🚀 No existing model found: running full initial fit (`fit`)...")
    classifier = SGDClassifier(
        loss='log_loss',
        max_iter=2000,
        tol=1e-2,
        class_weight='balanced',
        random_state=RANDOM_STATE
    )
    
    classifier.fit(train_embeddings, X_train_df['manual_label'])
    training_mode = "initial_full_fit"

fit_seconds = time.perf_counter() - fit_started

# Holdout evaluation
test_preds = classifier.predict(test_embeddings)
test_acc = accuracy_score(X_test_df['manual_label'], test_preds)
test_macro_f1 = f1_score(X_test_df['manual_label'], test_preds, average='macro')

print(f"\n================ EVALUATION REPORT ({training_mode}) ================")
print(classification_report(X_test_df['manual_label'], test_preds, zero_division=0))


🧠 Existing model found: running incremental update (`partial_fit`)...

================ EVALUATION REPORT (incremental_partial_fit) ================
                               precision    recall  f1-score   support

AI, cybersecurity and defense       0.62      0.68      0.65        22
    AI, environment and ocean       0.67      0.73      0.70        30
                      Core AI       0.84      0.79      0.81       105
                     No class       0.80      0.81      0.80       124

                     accuracy                           0.78       281
                    macro avg       0.73      0.75      0.74       281
                 weighted avg       0.79      0.78      0.78       281



In [19]:
# ==========================================
# 6. Rebuild Sub-axis References (with Dynamic Cache Invalidating)
# ==========================================
print("🔄 Processing pillar sub-axis reference embeddings...")

# Empreinte basée sur le contenu exact des sous-axes et le nom du modèle
subaxis_fingerprint_data = {
    "model_name": MODEL_NAME,
    "sequoia_saxes": SEQUOIA_SAXES,
}
current_subaxis_fingerprint = hashlib.sha256(
    json.dumps(subaxis_fingerprint_data, ensure_ascii=False, sort_keys=True).encode("utf-8")
).hexdigest()

recompute_subaxis = True

if SUBAXIS_PATH.exists():
    try:
        cached_data = joblib.load(SUBAXIS_PATH)
        # Vérification si le cache contient la même empreinte
        if isinstance(cached_data, dict) and cached_data.get("_fingerprint") == current_subaxis_fingerprint:
            subaxis_references = cached_data["references"]
            recompute_subaxis = False
            print(f"✅ Loaded valid cached sub-axis references from {SUBAXIS_PATH}")
    except Exception as e:
        print(f"⚠️ Failed to read cache, recomputing... ({e})")

if recompute_subaxis:
    print("🔄 Cache miss or sub-axes modified. Recomputing sub-axis embeddings...")
    subaxis_references = {}
    for pillar, subaxes in SEQUOIA_SAXES.items():
        subaxis_names = list(subaxes)
        subaxis_texts = [
            f"{' '.join(keywords)}"
            for keywords in subaxes.values()
        ]

        subaxis_references[pillar] = {
            'names': subaxis_names,
            'embeddings': encoder.encode(
                subaxis_texts,
                normalize_embeddings=True,
                show_progress_bar=False
            ),
        }
    
    # Sauvegarde des références accompagnées de leur empreinte
    joblib.dump(
        {
            "_fingerprint": current_subaxis_fingerprint,
            "references": subaxis_references
        },
        SUBAXIS_PATH
    )
    print(f"💾 Computed and saved updated sub-axis references to {SUBAXIS_PATH}")

🔄 Processing pillar sub-axis reference embeddings...
✅ Loaded valid cached sub-axis references from mpnet_sgd_artifacts/subaxis_references.joblib


In [ ]:
# ==========================================
# 7. Model Promotion, MLflow Tracking & Export
# ==========================================
# Read current best metric baseline (if exists)
previous_best_f1 = 0.0
if BEST_METRIC_PATH.exists():
    try:
        previous_best_f1 = json.loads(BEST_METRIC_PATH.read_text()).get("best_macro_f1", 0.0)
    except Exception:
        previous_best_f1 = 0.0

is_new_best = test_macro_f1 > previous_best_f1

with mlflow.start_run(run_name=f"pipeline-{training_mode}"):
    mlflow.log_params({
        "dataset_rows": len(df),
        "learning_mode": training_mode,
        "loss": "log_loss",
        "dvc_tracked": True,
        "embedding_cache_hit": int(cache_hit),
        "random_state": RANDOM_STATE
    })

    mlflow.log_metrics({
        "holdout_accuracy": test_acc,
        "holdout_macro_f1": test_macro_f1,
        "previous_best_macro_f1": previous_best_f1,
        "is_new_best": int(is_new_best),
        "embedding_time_sec": embedding_seconds,
        "fit_time_sec": fit_seconds
    })

    if is_new_best:
        print(f"🎯 NEW BEST MODEL! Macro-F1 improved from {previous_best_f1:.4f} to {test_macro_f1:.4f}. Saving artifacts & registering...")

        # 1. Save model weights to disk for server deployment
        joblib.dump(classifier, CLASSIFIER_PATH)
        encoder.save(str(ENCODER_PATH))

        # 2. Update benchmark state
        BEST_METRIC_PATH.write_text(json.dumps({"best_macro_f1": test_macro_f1}, indent=2))

        # 3. Update deployment metadata
        metadata = {
            'model_name': MODEL_NAME,
            'classifier': 'sklearn.linear_model.SGDClassifier',
            'loss': 'log_loss',
            'labels': CANONICAL_LABELS,
            'text_columns': text_columns,
            'subaxis_source': 'hal_sequoia_extractor.SEQUOIA_SAXES',
            'random_state': RANDOM_STATE,
            'last_training_mode': training_mode,
            'total_samples': len(df),
            'best_holdout_macro_f1': test_macro_f1
        }
        METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

        # 4. Register in MLflow Registry
        mlflow.sklearn.log_model(
            classifier,
            name="model",
            registered_model_name=REGISTERED_MODEL_NAME
        )
        print(f"🚀 Artifacts successfully updated and exported to: {ARTIFACT_DIR}")

    else:
        print(f"🛑 Candidate Macro-F1 ({test_macro_f1:.4f}) did not beat current best ({previous_best_f1:.4f}). Disk artifacts NOT updated.")
        # Log to MLflow run without registering or replacing production disk artifacts
        mlflow.sklearn.log_model(classifier, name="candidate_model")

🛑 Candidate Macro-F1 (0.7419) did not beat current best (0.7444). Disk artifacts NOT updated.


In [44]:
PILLAR_MAP = {
    "core ai": "Core AI",
    "ai, cybersecurity and defense": "AI, Cybersecurity and Defense",
    "ai, cybersecurity & defense": "AI, Cybersecurity and Defense",
    "ia & cybersécurité": "AI, Cybersecurity and Defense",
    "ai, environment and ocean": "AI, Environment and Ocean",
    "ai, environment & ocean": "AI, Environment and Ocean",
    "ia & environnement": "AI, Environment and Ocean",
    "no class": "No class",
    "non classifié": "No class",
}

def predict_article(title='', abstract='', keywords='', journal=''):
    text = ' '.join(str(value or '') for value in [title, abstract, keywords, journal]).strip()
    article_embedding = encoder.encode([text], normalize_embeddings=True)

    pillar_probabilities = classifier.predict_proba(article_embedding)[0]
    pillar_index = int(pillar_probabilities.argmax())
    pillar = str(classifier.classes_[pillar_index])

    print(f'Predicted pillar: {pillar} (confidence: {pillar_probabilities[pillar_index]:.3f})')
    
    # Unwrap subaxis_references if stored inside a fingerprint dict
    references_dict = subaxis_references.get("references", subaxis_references) if isinstance(subaxis_references, dict) else subaxis_references

    # Case-insensitive lookup using normalized key
    normalized_pillar = pillar.strip().lower()
    saxes_key = PILLAR_MAP.get(normalized_pillar, pillar)

    if saxes_key not in references_dict:
        print(f'Warning: No subaxis references found for pillar "{pillar}" (mapped key: "{saxes_key}"). Returning only pillar prediction.')
        return {
            'pillar': pillar,
            'pillar_confidence': float(pillar_probabilities[pillar_index]),
            'subaxis': 'No class',
            'subaxis_similarity': None,
            'pillar_probabilities': {str(label): float(score) for label, score in zip(classifier.classes_, pillar_probabilities)},
        }

    reference = references_dict[saxes_key]
    
    # Calculate similarity against subaxis embeddings
    subaxis_scores = encoder.similarity(article_embedding, reference['embeddings']).cpu().numpy()[0]
    subaxis_index = int(subaxis_scores.argmax())
    
    # Convert scores to probabilities via Softmax
    subaxis_probabilities = np.exp(subaxis_scores) / np.sum(np.exp(subaxis_scores))

    return {
        'pillar': pillar,
        'pillar_confidence': float(pillar_probabilities[pillar_index]),
        'subaxis': reference['names'][subaxis_index],
        'subaxis_similarity': float(subaxis_scores[subaxis_index]),
        'pillar_probabilities': {str(label): float(score) for label, score in zip(classifier.classes_, pillar_probabilities)},
        'subaxis_probabilities': {str(label): float(score) for label, score in zip(reference['names'], subaxis_probabilities)}
    }
# Test inference
predict_article(
    title='AI-based intrusion detection for secure networks',
    abstract='We study anomaly detection and cyber defense using machine learning.',
    keywords='cybersecurity, intrusion detection, anomaly detection',
)

Predicted pillar: AI, cybersecurity and defense (confidence: 0.951)


{'pillar': 'AI, cybersecurity and defense',
 'pillar_confidence': 0.9507308006286621,
 'subaxis': 'AI for Cybersecurity',
 'subaxis_similarity': 0.6881067752838135,
 'pillar_probabilities': {'AI, cybersecurity and defense': 0.9507308006286621,
  'AI, environment and ocean': 0.011019853875041008,
  'Core AI': 0.011157450266182423,
  'No class': 0.027091793715953827},
 'subaxis_probabilities': {'AI for Cybersecurity': 0.2668815851211548,
  'AI Security': 0.19800102710723877,
  'Intelligence and Information Security': 0.17302176356315613,
  'Intelligent and Secure Networks': 0.18488430976867676,
  'Autonomous Robotics and Interaction': 0.17721134424209595}}